In [ ]:
# ==========================================
# 1. Infrastructure Setup (Google Colab Environment)
# ==========================================
!pip uninstall -y datasets ragas pyarrow
!pip install --no-cache-dir datasets ragas pyarrow requests pandas langchain-ollama openai -q

import subprocess
import time
import os
import json
import requests
import pandas as pd
import ast 
import warnings
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextRecall
from openai import OpenAI
from ragas.llms import llm_factory
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import OllamaEmbeddings  

# Suppress terminal warnings
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("📥 Installing Ollama on the Colab Linux backend...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)

print("🚀 Launching Ollama Server in the background...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)  # Wait for server to initialize

print("🦙 Pulling llama3 model...")
subprocess.run(["ollama", "pull", "llama3"])

print("🧬 Pulling nomic-embed-text embedding model...")
subprocess.run(["ollama", "pull", "nomic-embed-text"])

print("✅ Ollama infrastructure setup complete!")

# ==========================================
# 2. Ragas Framework Configuration
# ==========================================
OLLAMA_MODEL_NAME = "llama3" 
OLLAMA_EMBED_NAME = "nomic-embed-text"
os.environ["OPENAI_API_KEY"] = "ollama-local-bypass"

# Setup local Ollama client via OpenAI v1 compatibility endpoint
ollama_client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
evaluator_llm = llm_factory(OLLAMA_MODEL_NAME, provider="openai", client=ollama_client)

# Wrap the Ollama Embedding model for Ragas compatibility
ollama_embed = OllamaEmbeddings(model=OLLAMA_EMBED_NAME, base_url="http://localhost:11434")
evaluator_embeddings = LangchainEmbeddingsWrapper(ollama_embed)

# Inject local LLM and Embeddings into metrics
metrics = [
    Faithfulness(llm=evaluator_llm),
    AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
    ContextRecall(llm=evaluator_llm)
]

print(f"🤖 Ollama Model Setup Complete! (LLM: {OLLAMA_MODEL_NAME}, Embed: {OLLAMA_EMBED_NAME})")

# Backend API Configuration
API_URL = "http://localhost:8000/ask"
CHECKPOINT_FILE = 'progress_checkpoint.csv'

# ==========================================
# 3. Data Loading & Resuming Logic
# ==========================================
if os.path.exists('golden_set.json'):
    with open('golden_set.json', 'r', encoding='utf-8') as f:
        golden_set = json.load(f)
    # Scope evaluation to the first 20 fact-based questions (Q01~Q20)
    golden_set = [item for item in golden_set if int(item['id'].replace('Q', '')) <= 20]
    print(f"🎯 Selected {len(golden_set)} validated questions for this benchmark.")
else:
    golden_set = []

if os.path.exists(CHECKPOINT_FILE):
    df_progress = pd.read_csv(CHECKPOINT_FILE)
    evaluation_data = df_progress.to_dict(orient='records')
    for item in evaluation_data:
        if isinstance(item['contexts'], str):
            try:
                item['contexts'] = ast.literal_eval(item['contexts'])
            except:
                item['contexts'] = [item['contexts']]
    completed_ids = set(df_progress['id'].tolist())
    print(f"🔄 Checkpoint found! Skipping {len(completed_ids)} items and resuming...")
else:
    evaluation_data = []
    completed_ids = set()

# ==========================================
# 4. Chatbot Response Collection
# ==========================================
print("🚀 Starting RAG Benchmark Evaluation...")
for item in golden_set:
    q_id = item['id']
    query = item['question']
    
    if q_id in completed_ids:
        continue
        
    print(f"Testing {q_id}: {query[:40]}...")
    try:
        response = requests.post(API_URL, json={"query": query}, timeout=120)
        if response.status_code == 200:
            data = response.json()
            retrieved_contexts = data.get('sources', [])
            chatbot_answer = data.get('answer', '')
            
            # Format raw contexts into standard List[str]
            clean_contexts = []
            if isinstance(retrieved_contexts, list):
                for ctx in retrieved_contexts:
                    if isinstance(ctx, dict):
                        clean_contexts.append(ctx.get('text', ctx.get('page_content', str(ctx))))
                    else:
                        clean_contexts.append(str(ctx))
            else:
                clean_contexts = [str(retrieved_contexts)]
            
            new_row = {
                "id": q_id,
                "question": query,
                "answer": chatbot_answer,
                "contexts": clean_contexts,
                "ground_truth": item['ground_truth']
            }
            evaluation_data.append(new_row)
            completed_ids.add(q_id)
            
            pd.DataFrame(evaluation_data).to_csv(CHECKPOINT_FILE, index=False)
        else:
            print(f"❌ Error at {q_id}: Status Code {response.status_code}.")
    except Exception as e:
        print(f"💥 Request failed at {q_id}: {e}.")

# ==========================================
# 5. Dataset Standardization & Ragas Scoring
# ==========================================
if not evaluation_data:
    print("🚨 No evaluation data collected.")
else:
    df_eval = pd.DataFrame(evaluation_data)
    
    # Map varying backend schemas to Ragas standard columns
    rename_map = {'user_input': 'question', 'query': 'question', 'response': 'answer', 'sources': 'contexts'}
    df_eval = df_eval.rename(columns={k: v for k, v in rename_map.items() if k in df_eval.columns})
    
    df_eval_final = df_eval[["question", "answer", "contexts", "ground_truth"]]
    dataset = Dataset.from_pandas(df_eval_final)

    print(f"\n📊 Calculating RAGAS Scores for {len(df_eval_final)} items using local Ollama...")
    try:
        result = evaluate(
            dataset=dataset,
            metrics=metrics,
            llm=evaluator_llm,
            embeddings=evaluator_embeddings
        )

        print("\n=== Final Evaluation Results ===")
        print(result)

        df_result = result.to_pandas()
        df_result.to_csv('rag_evaluation_report.csv', index=False)
        print("✅ Evaluation complete! Report saved to 'rag_evaluation_report.csv'")
        
    except Exception as eval_err:
        print(f"🚨 An error occurred during RAGAS evaluation: {eval_err}")